# BERT Training for Italian Twitter Stance Classification
This notebook prepares enriched text data and trains a BERT model using K-Fold cross-validation.

It includes logging with Weights & Biases and TensorBoard.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("cleaned_stance_dataset_enriched.csv")

# Create enriched text field
df["text"] = (
    "Content: " + df["content"].astype(str) + " | " +
    "Source: " + df["source"].astype(str) + " | " +
    "Sentiment: " + df["sentiment"].astype(str) + " | " +
    "Impressions: " + df["social impressions"].astype(str) + " | " +
    "Emojis: " + df["EMOJI_COUNT"].astype(str) + " | " +
    "LinkPresent: " + df["LINK present"].astype(str) + " | " +
    "BioSentiment: " + df["bio_sentiment"].astype(str)
)

# Select final dataset
df = df[["text", "stance_name.1"]].rename(columns={"stance_name.1": "label"})
df = df.dropna()
df.head()


In [ ]:
from sklearn.model_selection import KFold
from datasets import Dataset
from transformers import AutoTokenizer

model_name = "nickprock/twitter-xlm-roberta-italian-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Encode dataset
hf_dataset = Dataset.from_pandas(df)

def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import numpy as np
import wandb

# Encode labels
le = LabelEncoder()
tokenized_dataset = tokenized_dataset.add_column("label_id", le.fit_transform(tokenized_dataset["label"]))

# Setup wandb
wandb.init(project="bert-stance-classification", name="kfold-run")

# Define training args
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    load_best_model_at_end=True,
    report_to=["wandb", "tensorboard"],
)

# K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
for train_idx, val_idx in kf.split(tokenized_dataset):
    print(f"Training fold {fold}")
    fold += 1

    train_ds = tokenized_dataset.select(train_idx)
    val_ds = tokenized_dataset.select(val_idx)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(le.classes_))

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
    )

    trainer.train()
    preds = trainer.predict(val_ds)
    pred_labels = np.argmax(preds.predictions, axis=1)
    print(classification_report(val_ds['label_id'], pred_labels, target_names=le.classes_))
